# 6.2.图像卷积

上节我们解析了卷积层的原理，现在我们看看它的实际应用。由于卷积神经网络的设计是用于探索图像数据，本节我们将以图像为例。

本节中涉及反向传播算子的实现，但读者不必完全理解反向算子的推导，因为本节重点是前向计算与训练流程，只需知道反向的输入输出即可。

---

## 环境准备


In [ ]:
%matplotlib inline
import os
os.environ["TILE_FWK_DEVICE_ID"] = "0"

import pypto
import torch
from torch import nn

---

## 6.2.1.互相关运算

严格来说，卷积层是个错误的叫法，因为它所表达的运算其实是*互相关运算*（cross-correlation），而不是卷积运算。 根据 [<span style="color: #ffbf00; font-weight: bold;"> 6.1节 </span>](06.01_why_conv.ipynb)中的描述，在卷积层中，输入张量和核张量通过互相关运算产生输出张量。

首先，我们暂时忽略通道（第三维）这一情况，看看如何处理二维图像数据和隐藏表示。在 <span style="color: #ffbf00; font-weight: bold;"> 图6.2.1 </span>中，输入是高度为$3$、宽度为$3$的二维张量（即形状为$3 \times 3$）。卷积核的高度和宽度都是$2$，而卷积核窗口（或卷积窗口）的形状由内核的高度和宽度决定（即$2 \times 2$）。

<div style="border: solid 16px #f1f1f8; text-align: center; background-color: #f6f7f9">
<img src="./images/correlation.svg">
<p style="margin: 12px 0 4px 0; font-size: 0.9em; color: #555; text-align: center;">图6.2.1 二维互相关运算。阴影部分是第一个输出元素，以及用于计算输出的输入张量元素和核张量元素：</br>0 &times; 0 + 1 &times; 1 + 3 &times; 2 + 4 &times; 3 = 19.</p>
</div>
<br />

在二维互相关运算中，卷积窗口从输入张量的左上角开始，从左到右、从上到下滑动。 当卷积窗口滑动到新一个位置时，包含在该窗口中的部分张量与卷积核张量进行按元素相乘，得到的张量再求和得到一个单一的标量值，由此我们得出了这一位置的输出张量值。 在如上例子中，输出张量的四个元素由二维互相关运算得到，这个输出高度为$2$、宽度为$2$，如下所示：

$$
0\times0+1\times1+3\times2+4\times3=19,\\ 1\times0+2\times1+4\times2+5\times3=25,\\ 3\times0+4\times1+6\times2+7\times3=37,\\ 4\times0+5\times1+7\times2+8\times3=43. \tag{6.2.1}$$

注意，输出大小略小于输入大小。这是因为卷积核的宽度和高度大于1， 而卷积核只与图像中每个大小完全适合的位置进行互相关运算。 所以，输出大小等于输入大小$n_h \times n_w$减去卷积核大小$k_h \times k_w$，即：

$$(n_h-k_h+1) \times (n_w-k_w+1). \tag{6.2.2}$$

这是因为我们需要足够的空间在图像上“移动”卷积核。稍后，我们将看到如何通过在图像边界周围填充零来保证有足够的空间移动卷积核，从而保持输出大小不变。 接下来，我们在`corr2d`函数中实现如上过程，该函数接受输入张量`X`和卷积核张量`K`，并返回输出张量`Y`。

$$
\begin{array}{c|c}
\text{设定} &
\begin{array}{l}
X\in\mathbb{R}^{H\times W},\quad
K\in\mathbb{R}^{h\times w},\quad
Y\in\mathbb{R}^{(H-h+1)\times(W-w+1)} \\[4pt]
g=\partial L/\partial Y
\end{array}
\\ \hline
\text{前向} &
\displaystyle
Y_{i,j}=
\sum_{a=0}^{h-1}\sum_{b=0}^{w-1}
X_{i+a,j+b}\,K_{a,b}
\\[14pt] \hline
\text{反向} &
\displaystyle
\frac{\partial L}{\partial X_{p,q}}=
\sum_{i=0}^{H-h}\sum_{j=0}^{W-w}
g_{i,j}\,K_{p-i,q-j}
\quad (\text{valid范围内}) \\[14pt]
&
\displaystyle
\frac{\partial L}{\partial K_{a,b}}=
\sum_{i=0}^{H-h}\sum_{j=0}^{W-w}
g_{i,j}\,X_{i+a,j+b}
\end{array}
$$

In [3]:
@pypto.frontend.function
def corr2d_kernel(
    input:  pypto.Tensor([], pypto.DT_FP32),
    kernel: pypto.Tensor([], pypto.DT_FP32),
    output: pypto.Tensor([], pypto.DT_FP32),
):
    kH, kW = kernel.shape[0], kernel.shape[1]
    H_out, W_out = input.shape[0] - kH + 1, input.shape[1] - kW + 1
    pypto.set_vec_tile_shapes(8, 8)
    for i_idx in pypto.loop(H_out, name="LOOP_L0_i", idx_name="i_idx"):
        for j_idx in pypto.loop(W_out, name="LOOP_L1_j", idx_name="j_idx"):
            pypto.set_vec_tile_shapes(8, 8)
            window = input[i_idx:i_idx + kH, j_idx:j_idx + kW]
            mul_result = pypto.mul(window, kernel)
            sum_h = pypto.sum(mul_result, dim=0, keepdim=False)
            scalar = pypto.sum(sum_h, dim=0, keepdim=False)
            result_1x1 = pypto.reshape(scalar, [1, 1])
            pypto.assemble(result_1x1, [i_idx, j_idx], output)

@pypto.frontend.jit()
def corr2d_forward_kernel(
    X: pypto.Tensor([], pypto.DT_FP32),
    K: pypto.Tensor([], pypto.DT_FP32),
    Y: pypto.Tensor([], pypto.DT_FP32),
):
    corr2d_kernel(X, K, Y)

@pypto.frontend.jit()
def corr2d_backward_k_kernel(
    grad_Y: pypto.Tensor([], pypto.DT_FP32),
    X:      pypto.Tensor([], pypto.DT_FP32),
    grad_K: pypto.Tensor([], pypto.DT_FP32),
):
    corr2d_kernel(X, grad_Y, grad_K)

@pypto.frontend.jit()
def corr2d_backward_x_kernel(
    grad_Y: pypto.Tensor([], pypto.DT_FP32),
    K:      pypto.Tensor([], pypto.DT_FP32),
    grad_X: pypto.Tensor([], pypto.DT_FP32),
):
    kH, kW = K.shape[0], K.shape[1]
    H_in, W_in = grad_X.shape[0], grad_X.shape[1]
    pypto.set_vec_tile_shapes(8, 8)

    idx_h = pypto.arange(kH - 1, -1, -1)   # K: rot180
    idx_w = pypto.arange(kW - 1, -1, -1)
    idx_h_2d = pypto.expand_clone(pypto.reshape(idx_h, [kH, 1]), [kH, kW])
    idx_w_2d = pypto.expand_clone(pypto.reshape(idx_w, [1, kW]), [kH, kW])
    K_rot = pypto.gather(pypto.gather(K, 0, idx_h_2d), 1, idx_w_2d)

    if kW == 1 and kH == 1:   # G: Padding
        padded = grad_Y
    else:
        if kW > 1:  # 水平方向：左 kW-1 列 0 + grad_Y + 右 kW-1 列 0
            H_out = grad_Y.shape[0]
            zero_w = pypto.zeros(H_out, kW - 1, dtype=pypto.DT_FP32)
            padded_w = pypto.concat([zero_w, grad_Y, zero_w], dim=1)
        else:
            padded_w = grad_Y
        if kH > 1:  # 垂直方向：上 kH-1 行 0 + padded_w + 下 kH-1 行 0
            W_mid = padded_w.shape[1]
            zero_h = pypto.zeros(kH - 1, W_mid, dtype=pypto.DT_FP32)
            padded = pypto.concat([zero_h, padded_w, zero_h], dim=0)
        else:
            padded = padded_w

    corr2d_kernel(padded, K_rot, grad_X)

In [4]:
class PyPTOCorr2DFunction(torch.autograd.Function):
    @staticmethod
    def forward(ctx, X, K):
        X_c = X.contiguous()
        K_c = K.contiguous()

        n, _, h, w = X_c.shape
        _, _, kH, kW = K_c.shape
        out_h = h - kH + 1
        out_w = w - kW + 1

        Y = torch.zeros((n, 1, out_h, out_w), dtype=X_c.dtype, device=X_c.device)
        K_2d = K_c[0, 0]

        for batch_idx in range(n):
            corr2d_forward_kernel(X_c[batch_idx, 0], K_2d, Y[batch_idx, 0])

        ctx.save_for_backward(X_c, K_c)
        return Y

    @staticmethod
    def backward(ctx, grad_output):
        X, K = ctx.saved_tensors
        grad_output_c = grad_output.contiguous()

        n, _, h, w = X.shape
        _, _, kH, kW = K.shape

        grad_X = torch.zeros_like(X)
        grad_K = torch.zeros_like(K)
        K_2d = K[0, 0]

        grad_K_acc = torch.zeros_like(K_2d)

        for batch_idx in range(n):
            corr2d_backward_x_kernel(
                grad_output_c[batch_idx, 0],
                K_2d,
                grad_X[batch_idx, 0],
            )

            grad_K_one = torch.zeros_like(K_2d)
            corr2d_backward_k_kernel(
                grad_output_c[batch_idx, 0],
                X[batch_idx, 0],
                grad_K_one,
            )
            grad_K_acc += grad_K_one

        grad_K[0, 0] = grad_K_acc
        return grad_X, grad_K

def corr2d(X, K):
    X_4d = X.unsqueeze(0).unsqueeze(0)   # [1, 1, H, W]
    K_4d = K.unsqueeze(0).unsqueeze(0)   # [1, 1, kH, kW]

    Y_4d = PyPTOCorr2DFunction.apply(X_4d, K_4d)
    return Y_4d[0, 0]

<details class="code-note" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f9f9fb; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠代码说明</summary>
  <div class="code-note-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <ul style="margin: 0; padding-left: 20px;">
      <li style="margin: 0 0 8px 0;"><code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">@pypto.frontend.function</code> 用于定义可被 JIT 内核内联展开的复用函数：</li>
      <p style="margin: 0 0 10px 0;">被 <code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">@function</code> 装饰的函数不会被编译为独立 kernel，而是作为子图在编译期直接嵌入调用者的 IR 中，避免额外的调用开销。</br><code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">corr2d_kernel</code> 定义为 <code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">@function</code> 后，前向和两个反向 kernel 都可以复用同一个互相关计算逻辑，仅替换 input 和 kernel 的角色即可。</p>
      <li style="margin: 0 0 8px 0;">这里 <code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">PyPTOCorr2DFunction</code> 的输入 <code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">X</code>、 <code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">K</code> 采用四维 <code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">(N, C, H, W)</code> 而非 d2l 原书的二维 <code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">(H, W)</code>，原因是对齐 PyTorch <code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">nn.Conv2d</code> 的张量约定。</li>
      <p style="margin: 0;"><code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">nn.Conv2d</code> 要求输入 <code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">(N, C_in, H, W)</code>、 卷积核 <code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">(C_out, C_in, kH, kW)</code>。 <code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">PyPTOConv2d</code> 继承 <code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">nn.Module</code>，其 <code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">weight</code> 形状是 <code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">(C_out, C_in, kH, kW)</code>， <code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">backward</code> 返回的 <code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">grad_K</code> 也必须保持 4D 才能回填到 <code style="font-family: Consolas, monospace; background-color: #f3f4f6; padding: 2px 5px; border-radius: 3px; color: #1f2937; font-size: 13px;">conv2d.weight.grad</code>，故此处保持四维以与 PyTorch 自定义算子接口无缝对接。</p>
    </ul>
  </div>
</details>

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f9f9fb; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0; font-family: Consolas, monospace;">def corr2d(X, K):  #@save
    """计算二维互相关运算"""
    h, w = K.shape
    Y = torch.zeros((X.shape[0] - h + 1, X.shape[1] - w + 1))
    for i in range(Y.shape[0]):
        for j in range(Y.shape[1]):
            Y[i, j] = (X[i:i + h, j:j + w] * K).sum()
    return Y</pre>
  </div>
</details>

通过 <span style="color: #ffbf00; font-weight: bold;"> 图6.2.1 </span>的输入张量`X`和卷积核张量`K`，我们来验证上述二维互相关运算的输出。



In [5]:
X = torch.tensor([[0.0, 1.0, 2.0], [3.0, 4.0, 5.0], [6.0, 7.0, 8.0]], device='npu:0')
K = torch.tensor([[0.0, 1.0], [2.0, 3.0]], device='npu:0')
corr2d(X, K)

tensor([[19., 25.],
        [37., 43.]], device='npu:0')

---

## 6.2.2.卷积层

卷积层对输入和卷积核权重进行互相关运算，并在添加标量偏置之后产生输出。 所以，卷积层中的两个被训练的参数是卷积核权重和标量偏置。 就像我们之前随机初始化全连接层一样，在训练基于卷积层的模型时，我们也随机初始化卷积核权重。

基于上面定义的`corr2d`函数实现二维卷积层。在`__init__`构造函数中，将`weight`和`bias`声明为两个模型参数。前向传播函数调用`corr2d`函数并添加偏置。

In [6]:
class PyPTOConv2d(nn.Module):
    def __init__(
        self,
        in_channels=1,
        out_channels=1,
        kernel_size=(1, 2),
        bias=False,
        device="npu:0",
        dtype=torch.float32,
    ):
        super().__init__()

        if bias:
            raise NotImplementedError(
                "当前实现不支持 bias，请使用 bias=False。"
            )
        if in_channels != 1 or out_channels != 1:
            raise ValueError(
                f"当前实现仅支持单通道输入输出（in_channels=1, out_channels=1），"
                f"实际传入 in_channels={in_channels}, out_channels={out_channels}。"
            )

        self.in_channels = in_channels
        self.out_channels = out_channels
        self.kernel_size = kernel_size

        self.weight = nn.Parameter(
            torch.rand(out_channels, in_channels, kernel_size[0], kernel_size[1], device=device, dtype=dtype)
        )

    def forward(self, x):
        return PyPTOCorr2DFunction.apply(x, self.weight)

<details class="original-text" style="border: 1px solid #e3e3ee; border-radius: 4px; margin: 20px 0; overflow: hidden;">
  <summary style="padding: 10px 14px; font-weight: 500; cursor: pointer; background-color: #f9f9fb; list-style: none; color: #374151; font-size: 14px; letter-spacing: 0.01em;">点击：查看/折叠原文</summary>
  <div class="original-content" style="padding: 14px; background-color: #ffffff; line-height: 1.65; color: #4b5563; font-size: 14px;">
    <pre style="background-color: transparent; color: #1f2937; border: 1px solid #e5e7eb; border-radius: 0; font-size: 13px; padding: 12px; margin: 0; font-family: Consolas, monospace;">class Conv2D(nn.Module):
    def __init__(self, kernel_size):
        super().__init__()
        self.weight = nn.Parameter(torch.rand(kernel_size))
        self.bias = nn.Parameter(torch.zeros(1))
    def forward(self, x):
        return corr2d(x, self.weight) + self.bias</pre>
  </div>
</details>

高度和宽度分别为$h$和$w$的卷积核可以被称为$h \times w$卷积或$h \times w$卷积核。 我们也将带有$h \times w$卷积核的卷积层称为$h \times w$卷积层。

---

## 6.2.3.图像中目标的边缘检测

如下是卷积层的一个简单应用：通过找到像素变化的位置，来检测图像中不同颜色的边缘。 首先，我们构造一个$6\times 8$像素的黑白图像。中间四列为黑色（$0$），其余像素为白色（$1$）。



In [7]:
X = torch.ones((6, 8), device='npu:0')
X[:, 2:6] = 0
X

tensor([[1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.],
        [1., 1., 0., 0., 0., 0., 1., 1.]], device='npu:0')

接下来，我们构造一个高度为$1$、宽度为$2$的卷积核`K`。当进行互相关运算时，如果水平相邻的两元素相同，则输出为零，否则输出为非零。



In [8]:
K = torch.tensor([[1.0, -1.0]], device='npu:0')

现在，我们对参数`X`（输入）和`K`（卷积核）执行互相关运算。 如下所示，输出`Y`中的1代表从白色到黑色的边缘，-1代表从黑色到白色的边缘，其他情况的输出为$0$。



In [9]:
Y = corr2d(X, K)
Y

tensor([[ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.],
        [ 0.,  1.,  0.,  0.,  0., -1.,  0.]], device='npu:0')

现在我们将输入的二维图像转置，再进行如上的互相关运算。 其输出如下，之前检测到的垂直边缘消失了。 不出所料，这个卷积核`K`只可以检测垂直边缘，无法检测水平边缘。



In [10]:
corr2d(X.t(), K)

tensor([[0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.],
        [0., 0., 0., 0., 0.]], device='npu:0')

---

## 6.2.4.学习卷积核

如果我们只需寻找黑白边缘，那么以上`[1, -1]`的边缘检测器足以。然而，当有了更复杂数值的卷积核，或者连续的卷积层时，我们不可能手动设计滤波器。那么我们是否可以学习由`X`生成`Y`的卷积核呢？

现在让我们看看是否可以通过仅查看“输入-输出”对来学习由`X`生成`Y`的卷积核。 我们先构造一个卷积层，并将其卷积核初始化为随机张量。接下来，在每次迭代中，我们比较`Y`与卷积层输出的平方误差，然后计算梯度来更新卷积核。为了简单起见，我们在此使用内置的二维卷积层，并忽略偏置。



In [11]:
# 构造一个二维卷积层，它具有1个输出通道和形状为（1，2）的卷积核
conv2d = PyPTOConv2d(1, 1, kernel_size=(1, 2), bias=False)

# 这个二维卷积层使用四维输入和输出格式（批量大小、通道、高度、宽度），
# 其中批量大小和通道数都为1
X = X.reshape((1, 1, 6, 8))
Y = Y.reshape((1, 1, 6, 7))
lr = 3e-2  # 学习率

for i in range(10):
    Y_hat = conv2d(X)
    l = (Y_hat - Y) ** 2
    conv2d.zero_grad()
    l.sum().backward()
    # 迭代卷积核
    conv2d.weight.data[:] -= lr * conv2d.weight.grad
    if (i + 1) % 2 == 0:
        print(f'epoch {i+1}, loss {l.sum():.3f}')

epoch 2, loss 9.456
epoch 4, loss 3.178
epoch 6, loss 1.185
epoch 8, loss 0.466
epoch 10, loss 0.188


在$10$次迭代之后，误差已经降到足够低。现在我们来看看我们所学的卷积核的权重张量。



In [12]:
conv2d.weight.data.reshape((1, 2))

tensor([[ 1.0356, -0.9467]], device='npu:0')

细心的读者一定会发现，我们学习到的卷积核权重非常接近我们之前定义的卷积核`K`。

---

## 6.2.5.互相关和卷积

回想一下我们在 [<span style="color: #ffbf00; font-weight: bold;"> 6.1节 </span>](06.01_why_conv.ipynb)中观察到的互相关和卷积运算之间的对应关系。 为了得到正式的*卷积*运算输出，我们需要执行 <span style="color: #ffbf00; font-weight: bold;"> 公式6.1.2 </span>中定义的严格卷积运算，而不是互相关运算。 幸运的是，它们差别不大，我们只需水平和垂直翻转二维卷积核张量，然后对输入张量执行*互相关*运算。

值得注意的是，由于卷积核是从数据中学习到的，因此无论这些层执行严格的卷积运算还是互相关运算，卷积层的输出都不会受到影响。 为了说明这一点，假设卷积层执行*互相关*运算并学习 <span style="color: #ffbf00; font-weight: bold;"> 图6.2.1 </span>中的卷积核，该卷积核在这里由矩阵$\mathbf{K}$表示。 假设其他条件不变，当这个层执行严格的*卷积*时，学习的卷积核$\mathbf{K}'$在水平和垂直翻转之后将与$\mathbf{K}$相同。 也就是说，当卷积层对 <span style="color: #ffbf00; font-weight: bold;"> 图6.2.1 </span>中的输入和$\mathbf{K}'$执行严格*卷积*运算时，将得到与互相关运算 <span style="color: #ffbf00; font-weight: bold;"> 图6.2.1 </span>中相同的输出。

为了与深度学习文献中的标准术语保持一致，我们将继续把“互相关运算”称为卷积运算，尽管严格地说，它们略有不同。 此外，对于卷积核张量上的权重，我们称其为*元素*。

---



## 6.2.6.特征映射和感受野

如在 [<span style="color: #ffbf00; font-weight: bold;"> 6.1.4节 </span>](06.01_why_conv.ipynb)中所述， <span style="color: #ffbf00; font-weight: bold;"> 图6.2.1 </span>中输出的卷积层有时被称为*特征映射*（feature map），因为它可以被视为一个输入映射到下一层的空间维度的转换器。 在卷积神经网络中，对于某一层的任意元素$x$，其*感受野*（receptive field）是指在前向传播期间可能影响$x$计算的所有元素（来自所有先前层）。

请注意，感受野可能大于输入的实际大小。让我们用 <span style="color: #ffbf00; font-weight: bold;"> 图6.2.1 </span>为例来解释感受野： 给定$2 \times 2$卷积核，阴影输出元素值$19$的感受野是输入阴影部分的四个元素。 假设之前输出为$\mathbf{Y}$，其大小为$2 \times 2$，现在我们在其后附加一个卷积层，该卷积层以$\mathbf{Y}$为输入，输出单个元素$z$。 在这种情况下，$\mathbf{Y}$上的$z$的感受野包括$\mathbf{Y}$的所有四个元素，而输入的感受野包括最初所有九个输入元素。 因此，当一个特征图中的任意元素需要检测更广区域的输入特征时，我们可以构建一个更深的网络。

---

## 小结

* 二维卷积层的核心计算是二维互相关运算。最简单的形式是，对二维输入数据和卷积核执行互相关操作，然后添加一个偏置。
* 我们可以设计一个卷积核来检测图像的边缘。
* 我们可以从数据中学习卷积核的参数。
* 学习卷积核时，无论用严格卷积运算或互相关运算，卷积层的输出不会受太大影响。
* 当需要检测输入特征中更广区域时，我们可以构建一个更深的卷积网络。

---

## 练习

1. 构建一个具有对角线边缘的图像`X`。
    1. 如果将本节中举例的卷积核`K`应用于`X`，会发生什么情况？
    2. 如果转置`X`会发生什么？
    3. 如果转置`K`会发生什么？
2. 在我们创建的`Conv2D`自动求导时，有什么错误消息？
3. 如何通过改变输入张量和卷积核张量，将互相关运算表示为矩阵乘法？
4. 手工设计一些卷积核。
    1. 二阶导数的核的形式是什么？
    2. 积分的核的形式是什么？
    3. 得到$d$次导数的最小核的大小是多少？



详细参考答案见 [answers/06.02_reference_answer](./answers/06.02_reference_answer.ipynb)。

#### 参考答案（PyPTO版）

In [ ]:
!cat answers/txt/06.02_reference_answer_pypto.txt

#### 参考答案（PyTorch版）

In [ ]:
!cat answers/txt/06.02_reference_answer_pytorch.txt